In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
path_to_data = '/content/drive/MyDrive/CEMS'
print(os.listdir(path_to_data))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['EMSN194', 'EMSR352', 'EMSR416', 'EMSR466', 'EMSR339', 'EMSR468', 'EMSR417']


In [8]:
!pip install rasterio

In [9]:
import numpy as np
import rasterio
from scipy.stats import wasserstein_distance

In [20]:
import numpy as np
import rasterio
from scipy.stats import wasserstein_distance

# ===== Training Distribution =====
TRAINING_STATS = {
    'S1_before_flood': {'mean': 1735.8619384765625, 'std': 3036.968505859375},
    'S1_after_flood':  {'mean': 1803.71923828125,  'std': 3154.760009765625},
    'Terrain':         {'mean': 24.5716609954834,  'std': 53.64265823364258},
    'LULC':            {'mean': 40.98617935180664, 'std': 18.569223403930664}
}


# ===== Helper Functions =====
def load_tiff(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)


def compute_stats(arr):
    return float(np.nanmean(arr)), float(np.nanstd(arr))


def drift_metrics(train_mean, train_std, inf_mean, inf_std):
    z_mean = abs(inf_mean - train_mean) / (train_std + 1e-6)
    z_std  = abs(inf_std - train_std)   / (train_std + 1e-6)
    return z_mean, z_std


# ===== Drift Check =====
def run_drift_check(
    s1_before_path,
    s1_after_path,
    terrain_path,
    lulc_path
):
    print("\n==== Loading TIFFs ====")
    s1_before = load_tiff(s1_before_path)   # 4 channels
    s1_after  = load_tiff(s1_after_path)    # 4 channels
    terrain   = load_tiff(terrain_path)     # 2 channels
    lulc      = load_tiff(lulc_path)        # 1 channel

    # Concatenate to (11, H, W)
    x = np.concatenate([s1_before, s1_after, terrain, lulc], axis=0)

    print("\n==== Computing Inference Stats ====")

    groups = {
        "S1_before_flood": x[0:4],
        "S1_after_flood":  x[4:8],
        "Terrain":         x[8:10],
        "LULC":            x[10:11]
    }

    report = {}

    for key, arr in groups.items():
        inf_mean, inf_std = compute_stats(arr)
        tr_mean, tr_std = TRAINING_STATS[key]['mean'], TRAINING_STATS[key]['std']

        z_m, z_s = drift_metrics(tr_mean, tr_std, inf_mean, inf_std)

        # Wasserstein distance normalized by training std
        w_dist = wasserstein_distance(
            arr.flatten() / (tr_std + 1e-6),
            np.full(arr.size, tr_mean) / (tr_std + 1e-6)
        )

        report[key] = {
            "inference_mean": inf_mean,
            "inference_std": inf_std,
            "z_mean_shift": float(z_m),
            "z_std_shift": float(z_s),
            "wasserstein_distance": float(w_dist)
        }

    print("\n==== Drift Report ====")
    for k, v in report.items():
        print(f"\n[{k}]")
        print(f" Mean shift (z-score): {v['z_mean_shift']:.4f}")
        print(f" Std shift  (z-score): {v['z_std_shift']:.4f}")
        print(f" Wasserstein Distance: {v['wasserstein_distance']:.4f}")
    # Define thresholds
    Z_THRESH = 2.0
    WD_THRESH = 1.0

    for k, v in report.items():
        drift_flag = (v['z_mean_shift'] > Z_THRESH) or (v['z_std_shift'] > Z_THRESH) or (v['wasserstein_distance'] > WD_THRESH)
        status = "🚨 DRIFT DETECTED" if drift_flag else "✅ No drift"
        print(f"\n[{k}] {status}")
        print(f" Mean shift (z-score): {v['z_mean_shift']:.4f}")
        print(f" Std shift  (z-score): {v['z_std_shift']:.4f}")
        print(f" Wasserstein Distance: {v['wasserstein_distance']:.4f}")

In [21]:
run_drift_check(
    "/content/drive/MyDrive/CEMS/EMSN194/s1_before_flood/000000_s1_before_flood.tif",
    "/content/drive/MyDrive/CEMS/EMSN194/s1_during_flood/000000_s1_during_flood.tif",
    "/content/drive/MyDrive/CEMS/EMSN194/terrain/000000_terrain.tif",
    "/content/drive/MyDrive/CEMS/EMSN194/LULC/000000_LULC.tif"
)


==== Loading TIFFs ====

==== Computing Inference Stats ====

==== Drift Report ====

[S1_before_flood]
 Mean shift (z-score): 0.1382
 Std shift  (z-score): 0.2344
 Wasserstein Distance: 0.9999

[S1_after_flood]
 Mean shift (z-score): 0.1312
 Std shift  (z-score): 0.2222
 Wasserstein Distance: 0.9928

[Terrain]
 Mean shift (z-score): 0.2612
 Std shift  (z-score): 0.7460
 Wasserstein Distance: 0.3411

[LULC]
 Mean shift (z-score): 0.7642
 Std shift  (z-score): 0.2917
 Wasserstein Distance: 1.3439

[S1_before_flood] ✅ No drift
 Mean shift (z-score): 0.1382
 Std shift  (z-score): 0.2344
 Wasserstein Distance: 0.9999

[S1_after_flood] ✅ No drift
 Mean shift (z-score): 0.1312
 Std shift  (z-score): 0.2222
 Wasserstein Distance: 0.9928

[Terrain] ✅ No drift
 Mean shift (z-score): 0.2612
 Std shift  (z-score): 0.7460
 Wasserstein Distance: 0.3411

[LULC] 🚨 DRIFT DETECTED
 Mean shift (z-score): 0.7642
 Std shift  (z-score): 0.2917
 Wasserstein Distance: 1.3439
